# HCLA RQ3–RQ5 Post-hoc Statistical Analysis

이 노트북은 추가 API 호출 없이 이미 저장된 CSV 결과만 읽어 다음을 수행합니다.

- RQ3: HCLA vs 각 baseline paired Wilcoxon + Holm correction
- RQ5: 10회 반복을 case 단위로 평균 집계 후 Friedman + pairwise Wilcoxon + Holm
- RQ4: 기존 n=147 원인 진단 + 3회 반복을 case 단위로 집계한 case-level 재분석


In [ ]:
import os, glob
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon

try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception:
    pass

BASE_CANDIDATES = [
    "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_SHAP",
    "/content/drive/MyDrive/Colab Notebooks/Results/HCLA_MultiModel_Qwen_Llama_OpenRouter_SHAP",
]
BASE_DIRS = [p for p in BASE_CANDIDATES if os.path.exists(p)]
if not BASE_DIRS:
    raise FileNotFoundError("결과 폴더를 찾지 못했습니다. BASE_CANDIDATES를 실제 경로에 맞게 수정하세요.")

def holm_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    out = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m-rank)*pvals[idx])
        out[idx] = min(1.0, running)
    return out

def safe_wilcoxon(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    try:
        stat, p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
    except ValueError:
        stat, p = 0.0, 1.0
    return float(stat), float(p), len(x)

def find_best_csv(filename, required_cols=None, prefer_rows=None):
    found = []
    for base in BASE_DIRS:
        for p in glob.glob(os.path.join(base, "**", filename), recursive=True):
            try:
                df = pd.read_csv(p)
                if required_cols and not set(required_cols).issubset(df.columns):
                    continue
                found.append((p, df))
            except Exception:
                pass
    if not found:
        raise FileNotFoundError(filename)
    if prefer_rows is None:
        found.sort(key=lambda x: -len(x[1]))
    else:
        found.sort(key=lambda x: (abs(len(x[1])-prefer_rows), -len(x[1])))
    print("Selected:", found[0][0], "| rows:", len(found[0][1]))
    return found[0]


## RQ3 — HCLA vs 각 baseline
동일한 50개 case를 4개 방법이 공유하므로 paired Wilcoxon을 사용합니다.
Holm 보정은 HCLA vs Numerical / Feature Attribution / Template의 3개 비교에 적용합니다.


In [ ]:
RQ3_METRICS = ["clarity","evidence_integration","calibrated_usefulness"]
METHODS = ["Numerical","Feature Attribution","Template","HCLA"]
BASELINES = ["Numerical","Feature Attribution","Template"]

rq3_path, rq3 = find_best_csv(
    "rq3_judgments.csv",
    ["case_id","method"] + RQ3_METRICS,
    200
)

rq3_case = rq3.groupby(["case_id","method"], as_index=False)[RQ3_METRICS].mean(numeric_only=True)
rows = []

for metric in RQ3_METRICS:
    pv = rq3_case.pivot(index="case_id", columns="method", values=metric).dropna(subset=METHODS)
    fstat, fp = friedmanchisquare(*[pv[m].values for m in METHODS])
    rows.append(dict(metric=metric,test="Friedman",comparison="All 4 methods",
                     statistic=fstat,p_raw=fp,p_holm=np.nan,n_cases=len(pv)))
    pairs = []
    for base in BASELINES:
        stat,p,n = safe_wilcoxon(pv["HCLA"], pv[base])
        pairs.append(dict(metric=metric,test="Wilcoxon",
                          comparison=f"HCLA vs {base}",statistic=stat,p_raw=p,n_cases=n))
    adj = holm_adjust([r["p_raw"] for r in pairs])
    for r,pa in zip(pairs,adj):
        r["p_holm"] = pa
        r["significant_holm_0.05"] = pa < .05
        rows.append(r)

rq3_stats = pd.DataFrame(rows)
out3 = os.path.join(os.path.dirname(rq3_path),"rq3_posthoc_wilcoxon_holm.csv")
rq3_stats.to_csv(out3,index=False,encoding="utf-8-sig")
display(rq3_stats)
print("Saved:", out3)


## RQ5 — 50 case 단위 집계 후 통계
500 rows/condition을 독립 표본으로 쓰지 않고, case×condition별 10회 반복 평균을 만든 뒤 50개 case를 분석 단위로 사용합니다.


In [ ]:
RQ5_CONDITIONS = ["Standard","Ambiguous","Adversarial"]
RQ5_METRICS = ["ESR","UCR","CR","constraint_adherence","semantic_consistency"]

rq5_path, rq5 = find_best_csv("rq5_robustness_full.csv",["case_id","condition"],1500)
metrics = [m for m in RQ5_METRICS if m in rq5.columns]

rq5_case = rq5.groupby(["case_id","condition"],as_index=False)[metrics].mean(numeric_only=True)
rows = []

for metric in metrics:
    pv = rq5_case.pivot(index="case_id",columns="condition",values=metric).dropna(subset=RQ5_CONDITIONS)
    fstat,fp = friedmanchisquare(*[pv[c].values for c in RQ5_CONDITIONS])
    rows.append(dict(metric=metric,test="Friedman",comparison="All 3 conditions",
                     statistic=fstat,p_raw=fp,p_holm=np.nan,n_cases=len(pv)))
    pairs=[]
    for a,b in combinations(RQ5_CONDITIONS,2):
        stat,p,n = safe_wilcoxon(pv[a],pv[b])
        pairs.append(dict(metric=metric,test="Wilcoxon",comparison=f"{a} vs {b}",
                          statistic=stat,p_raw=p,n_cases=n))
    adj=holm_adjust([r["p_raw"] for r in pairs])
    for r,pa in zip(pairs,adj):
        r["p_holm"]=pa
        r["significant_holm_0.05"]=pa<.05
        rows.append(r)

rq5_stats=pd.DataFrame(rows)
d5=os.path.dirname(rq5_path)
rq5_case.to_csv(os.path.join(d5,"rq5_case_level_aggregated.csv"),index=False,encoding="utf-8-sig")
rq5_stats.to_csv(os.path.join(d5,"rq5_caselevel_friedman_wilcoxon_holm.csv"),index=False,encoding="utf-8-sig")
display(rq5_stats)


## RQ4 — n=147 원인 진단 + case-level 재분석
기존 코드는 case_id×rep를 분석 단위로 두어 이론상 n=150입니다. 어떤 조건의 metric이 NaN이면 해당 paired row가 제거되어 n=147이 될 수 있습니다.

논문에서는 rep 3회를 독립 표본처럼 세지 않고, case×condition 내에서 반복을 평균낸 뒤 case를 분석 단위로 쓰는 편이 더 안전합니다.


In [ ]:
RQ4_CONDITIONS = ["Full_HCLA","Without_RoleSeparation","Without_EvidenceConstraint","Without_InteractiveRefinement"]
RQ4_METRICS = ["ESR","UCR","CR","clarity","trust_support","actionability"]

rq4_path,rq4 = find_best_csv("rq4_ablation_full.csv",["case_id","condition","rep"],600)
metrics=[m for m in RQ4_METRICS if m in rq4.columns]

# raw NaN rows
nan_rows=rq4.loc[rq4[metrics].isna().any(axis=1),["case_id","rep","condition"]+metrics].copy()
print("Rows with metric NaN:",len(nan_rows))
display(nan_rows)

# 기존 n=150 -> ? 원인
diag=[]
for metric in metrics:
    pv=rq4.pivot_table(index=["case_id","rep"],columns="condition",values=metric,aggfunc="mean")
    complete=pv.dropna(subset=RQ4_CONDITIONS)
    dropped=pv.index.difference(complete.index)
    diag.append(dict(
        metric=metric,
        possible_case_rep_pairs=len(pv),
        complete_case_rep_pairs=len(complete),
        dropped_pairs=len(pv)-len(complete),
        dropped_keys="; ".join([f"{cid}|rep={rep}" for cid,rep in dropped])
    ))
diag_df=pd.DataFrame(diag)
display(diag_df)

# 각 case-condition에서 valid rep 수
rep_counts=[]
for metric in metrics:
    t=rq4.groupby(["case_id","condition"])[metric].count().reset_index(name="valid_reps")
    t["metric"]=metric
    rep_counts.append(t)
rep_counts=pd.concat(rep_counts,ignore_index=True)
display(rep_counts[rep_counts.valid_reps<3])

# case 단위 집계
rq4_case=rq4.groupby(["case_id","condition"],as_index=False)[metrics].mean(numeric_only=True)
rows=[]
for metric in metrics:
    pv=rq4_case.pivot(index="case_id",columns="condition",values=metric).dropna(subset=RQ4_CONDITIONS)
    fstat,fp=friedmanchisquare(*[pv[c].values for c in RQ4_CONDITIONS])
    rows.append(dict(metric=metric,test="Friedman",comparison="All 4 conditions",
                     statistic=fstat,p_raw=fp,p_holm=np.nan,n_cases=len(pv)))
    pairs=[]
    for a,b in combinations(RQ4_CONDITIONS,2):
        stat,p,n=safe_wilcoxon(pv[a],pv[b])
        pairs.append(dict(metric=metric,test="Wilcoxon",comparison=f"{a} vs {b}",
                          statistic=stat,p_raw=p,n_cases=n))
    adj=holm_adjust([r["p_raw"] for r in pairs])
    for r,pa in zip(pairs,adj):
        r["p_holm"]=pa
        r["significant_holm_0.05"]=pa<.05
        rows.append(r)

rq4_stats=pd.DataFrame(rows)
d4=os.path.dirname(rq4_path)
diag_df.to_csv(os.path.join(d4,"rq4_n147_diagnostics.csv"),index=False,encoding="utf-8-sig")
nan_rows.to_csv(os.path.join(d4,"rq4_nan_rows.csv"),index=False,encoding="utf-8-sig")
rep_counts.to_csv(os.path.join(d4,"rq4_valid_rep_counts.csv"),index=False,encoding="utf-8-sig")
rq4_case.to_csv(os.path.join(d4,"rq4_case_level_aggregated.csv"),index=False,encoding="utf-8-sig")
rq4_stats.to_csv(os.path.join(d4,"rq4_caselevel_friedman_wilcoxon_holm.csv"),index=False,encoding="utf-8-sig")
display(rq4_stats)


## 해석 시 주의
- RQ3: Friedman + HCLA vs baseline 3개 post-hoc 결과를 함께 보고
- RQ5: 통계적 n은 1,500 row가 아니라 50 case
- RQ4: `rq4_n147_diagnostics.csv`로 147의 원인을 먼저 확인
- RQ4에서 valid_reps < 3가 있으면 가능하면 해당 judge 결과를 보충한 뒤 최종 통계를 다시 계산하는 것이 가장 깔끔합니다.
